# RAG 파이프라인 (Colab 베이스라인 서빙)

실행 전 확인: `런타임 > 런타임 유형 변경 > T4 GPU`로 설정.

구성: A 의존성 설치 → B 데이터 업로드 → C schemas → D indexing → E retriever → F augmentation → G generation → H FastAPI app → I 서버 실행 → J 테스트 요청

## A. 의존성 설치

In [ ]:
!pip install -q fastapi "uvicorn[standard]" pydantic torch transformers accelerate \
    langchain-core langchain-huggingface sentence-transformers nest-asyncio pyngrok

## B. retrospective.md 업로드

In [ ]:
from google.colab import files

uploaded = files.upload()
FILE_PATH = list(uploaded.keys())[0]
print(f"업로드된 파일: {FILE_PATH}")

## C. schemas

In [ ]:
import time
import uuid
from typing import Literal

from pydantic import BaseModel, Field


class ChatMessage(BaseModel):
    role: Literal["user", "assistant", "system"]
    content: str


class ChatCompletionRequest(BaseModel):
    model: str
    messages: list[ChatMessage]


class ChatCompletionResponseChoice(BaseModel):
    index: int = 0
    message: ChatMessage
    finish_reason: str = "stop"


class ChatCompletionResponse(BaseModel):
    id: str = Field(default_factory=lambda: f"chatcmpl-{uuid.uuid4().hex}")
    object: str = "chat.completion"
    created: int = Field(default_factory=lambda: int(time.time()))
    model: str
    choices: list[ChatCompletionResponseChoice]

## D. indexing (로딩 → 파싱 → 청킹 → 임베딩 → 벡터DB 저장)

In [ ]:
import re
from pathlib import Path

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
SECTION_HEADER_PATTERN = re.compile(r"(?=^### )", re.MULTILINE)


def build_vector_store() -> InMemoryVectorStore:
    # 1. 로딩
    markdown_text = Path(FILE_PATH).read_text(encoding="utf-8")

    # 2. 파싱
    raw_sections = SECTION_HEADER_PATTERN.split(markdown_text)

    # 3. 청킹
    section_texts = [section.strip() for section in raw_sections if section.strip().startswith("### ")]

    documents: list[Document] = []
    for section_text in section_texts:
        title_line = section_text.splitlines()[0]
        title = title_line.removeprefix("### ").strip()
        documents.append(Document(page_content=section_text, metadata={"title": title}))

    # 4. 임베딩
    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

    # 5. 벡터DB 저장
    vector_store = InMemoryVectorStore(embeddings)
    vector_store.add_documents(documents)

    return vector_store

## E. retriever

In [ ]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

TOP_K = 4


def retrieve_relevant_documents(vector_store: InMemoryVectorStore, query: str) -> list[Document]:
    return vector_store.similarity_search(query, k=TOP_K)

## F. augmentation (컨텍스트 구성 → 메시지 구성)

In [ ]:
SYSTEM_PROMPT_TEMPLATE = "아래 참고자료를 바탕으로 질문에 답하시오.\n\n{context}"


def build_augmented_messages(documents: list[Document], user_query: str) -> list[ChatMessage]:
    # 1. 컨텍스트 구성: 검색된 문서를 "[참고자료 N] 제목 + 본문" 형식으로 나열
    context_blocks = []
    for index, document in enumerate(documents, start=1):
        title = document.metadata["title"]
        context_blocks.append(f"[참고자료 {index}] {title}\n{document.page_content}")
    context_text = "\n\n".join(context_blocks)

    # 2. 메시지 구성: 컨텍스트를 담은 system 메시지 + 질문을 담은 user 메시지
    system_message = ChatMessage(role="system", content=SYSTEM_PROMPT_TEMPLATE.format(context=context_text))
    user_message = ChatMessage(role="user", content=user_query)

    return [system_message, user_message]

## G. generation

In [ ]:
from typing import Callable, cast

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, PreTrainedModel, PreTrainedTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
MAX_NEW_TOKENS = 512


def load_model_and_tokenizer() -> tuple[PreTrainedModel, PreTrainedTokenizer]:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto")
    return model, tokenizer


def generate_response(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizer,
    messages: list[ChatMessage],
) -> str:
    chat_messages = [{"role": message.role, "content": message.content} for message in messages]
    prompt_text = cast(str, tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True))
    input_ids = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    generate = cast(Callable[..., torch.LongTensor], model.generate)
    output_ids = generate(**input_ids, max_new_tokens=MAX_NEW_TOKENS)
    generated_ids = output_ids[0][input_ids["input_ids"].shape[1]:]
    response_text = cast(str, tokenizer.decode(generated_ids, skip_special_tokens=True))

    return response_text

## H. FastAPI app (lifespan + /v1/chat/completions)

In [ ]:
from contextlib import asynccontextmanager

from fastapi import FastAPI, Request


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.vector_store = build_vector_store()
    model, tokenizer = load_model_and_tokenizer()
    app.state.model = model
    app.state.tokenizer = tokenizer
    yield


app = FastAPI(lifespan=lifespan)


@app.post("/v1/chat/completions")
def chat_completions(chat_request: ChatCompletionRequest, http_request: Request) -> ChatCompletionResponse:
    user_query = chat_request.messages[-1].content

    documents = retrieve_relevant_documents(http_request.app.state.vector_store, user_query)
    augmented_messages = build_augmented_messages(documents, user_query)
    response_text = generate_response(
        http_request.app.state.model,
        http_request.app.state.tokenizer,
        augmented_messages,
    )

    response_message = ChatMessage(role="assistant", content=response_text)
    choice = ChatCompletionResponseChoice(message=response_message)
    return ChatCompletionResponse(model=chat_request.model, choices=[choice])

## I. 서버 실행

Colab은 셀 하나가 끝나야 다음 셀이 실행되므로, 서버를 백그라운드 스레드로 띄워야 같은 노트북에서 테스트 요청까지 이어서 실행 가능함.

In [ ]:
import time
from threading import Thread

import nest_asyncio
import requests
import uvicorn

nest_asyncio.apply()


def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)


server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

for _ in range(120):
    time.sleep(2)
    try:
        requests.get("http://127.0.0.1:8000/docs", timeout=1)
        print("서버 준비 완료")
        break
    except requests.exceptions.ConnectionError:
        continue
else:
    print("서버가 240초 안에 준비되지 않음. 아래 로그 확인 필요")

## J. 테스트 요청

In [ ]:
import requests

response = requests.post(
    "http://127.0.0.1:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-3B-Instruct",
        "messages": [{"role": "user", "content": "docker run 관련 트러블슈팅 있었나?"}],
    },
)
print(response.status_code)
print(response.json())